In [4]:
from pathlib import Path
import os, binascii, textwrap

PARQUET = Path("../data/features/features_202507.parquet").resolve()
print("Ruta absoluta:", PARQUET)
print("Existe:", PARQUET.exists(), "· Tamaño:", PARQUET.stat().st_size, "bytes")

# 1) Inspecciona los 8 primeros bytes
with PARQUET.open("rb") as f:
    magic = f.read(8)
print("Cabecera          :", binascii.hexlify(magic))

# 2) Inspecciona los 8 últimos bytes
with PARQUET.open("rb") as f:
    f.seek(-8, os.SEEK_END)
    magic_tail = f.read(8)
print("Pie de fichero    :", binascii.hexlify(magic_tail))

# 3) Intenta leer como gzip (por si quedó comprimido accidentalmente)
import gzip, io
try:
    with gzip.open(PARQUET, "rb") as gz:
        head = gz.read(4)
    print("» ¡Parece gzip! Primeros 4 bytes descomprimidos:", head)
except OSError:
    print("» No es gzip")


Ruta absoluta: D:\Dev\Python\memebot3\data\features\features_202507.parquet
Existe: True · Tamaño: 7653060 bytes
Cabecera          : b'0000000000000000'
Pie de fichero    : b'0000000000000000'
» No es gzip


In [ ]:
label_counts = df["label"].value_counts().sort_index()
display(label_counts.to_frame("count"))

pos_ratio = label_counts.get(1, 0) / label_counts.sum()
print(f"Proporción positiva: {pos_ratio:.2%}")


In [ ]:
df = df.sort_values("timestamp")
cut = int(len(df)*0.8)
train_df = df.iloc[:cut]
valid_df = df.iloc[cut:]


In [ ]:
ratio = 0.25            # samplea solo el 25 % de la clase 0
train_under = (train_df
               .groupby("label", group_keys=False)
               .apply(lambda x: x.sample(frac=ratio if x.name==0 else 1,
                                         random_state=42)))


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV

X_cols = [c for c in df.columns
          if c not in ("label","pnl","timestamp","ts","pnl_pct","address","discovered_via")]

grid = {
    "num_leaves":   [31, 63],
    "learning_rate":[0.05, 0.1],
    "max_depth":    [-1, 6, 12],
    "min_data_in_leaf":[20, 100],
    "feature_fraction":[0.8, 1.0],
}

clf = LGBMClassifier(objective="binary",
                     n_estimators=400,
                     scale_pos_weight=len(train_under[train_under.label==0]) /
                                       len(train_under[train_under.label==1]),
                     n_jobs=-1)

gs = GridSearchCV(clf, grid, cv=3, scoring="roc_auc", verbose=1)
gs.fit(train_under[X_cols], train_under["label"])

print("Mejor AUC CV:", gs.best_score_)
best = gs.best_estimator_


In [ ]:
from sklearn.metrics import precision_recall_curve

proba_valid = best.predict_proba(valid_df[X_cols])[:,1]
prec, rec, thr = precision_recall_curve(valid_df["label"], proba_valid)

# fija p.ej. precisión ≥ 0.80
target_prec = 0.80
thr_opt = next(t for p,r,t in zip(prec,rec,thr) if p >= target_prec)
print(f"Threshold @{target_prec:.0%} precisión ≈ {thr_opt:.2f}")
